modified for few words



In [5]:
# --- States ---
q0, qS, qES, qIES, qSG = "q0", "qS", "qES", "qIES", "qSG"
qFAIL = "qFAIL" # The trap state for invalid words

alphabet = [chr(i) for i in range(ord('a'), ord('z')+1)]
sibilants = ['s', 'x', 'z', 'h'] # 'h' covers 'ch' and 'sh' stems

# --- Delta (Transition) Table ---
delta = {}

# State q0: Initial state
for ch in alphabet:
    delta[(q0, ch)] = qS if ch == 's' else qSG

# State qS: Has seen a final 's'
for ch in alphabet:
    if ch == 'e':
        delta[(qS, ch)] = qES
    elif ch in sibilants or ch == 'y':
        delta[(qS, ch)] = qFAIL
    else:
        delta[(qS, ch)] = qSG

# State qES: Has seen 'es'
for ch in alphabet:
    delta[(qES, ch)] = qIES if ch == 'i' else qSG

# States qIES and qSG
for ch in alphabet:
    delta[(qIES, ch)] = qSG
    delta[(qSG, ch)] = qSG

# Make the trap state "sticky"
for ch in alphabet:
    delta[(qFAIL, ch)] = qFAIL


# --- Output Function (Lambda) ---
lam = {}
all_states = [q0, qS, qES, qIES, qSG, qFAIL]
for state in all_states:
    for ch in alphabet:
        lam[(state, ch)] = ch

lam[(q0, 's')] = ''
lam[(qS, 'e')] = ''

# Rules for stems after 'es' has been seen
lam[(qES, 's')] = 's'    # buses -> bus
lam[(qES, 'z')] = 'z'    # waltzes -> waltz
lam[(qES, 'x')] = 'x'    # boxes -> box
lam[(qES, 'h')] = 'h'    # bushes -> bush
lam[(qES, 'i')] = 'y'    # cities -> city

# The fix for the "homes" -> "hoem" bug
for ch in alphabet:
    if ch not in ['i', 's', 'z', 'x', 'h']:
        # Pre-reverse the output to counteract the final reversal
        lam[(qES, ch)] = ch + 'e'


# --- Transduce Function ---
def transduce(word, start_state=q0):
    """
    Transduces a word using the FST, processing it in reverse.
    Returns (stem, is_plural) or (word, "INVALID") for invalid words.
    """
    state = start_state
    output_chars = []

    for ch in reversed(word):
        if (state, ch) not in delta:
            return word, "INVALID"

        output = lam.get((state, ch), ch)
        state = delta[(state, ch)]
        output_chars.append(output)

    if state == qFAIL:
        return word, "INVALID"

    stem = "".join(reversed(output_chars))
    is_plural = (stem != word)

    return stem, is_plural

# --- Main Execution Loop ---
def process_corpus(filename="brown_nouns.txt"):
    """
    Reads words from a file, processes them with the FST,
    and prints the analysis.
    """
    print(f"--- FST Analysis of Nouns from {filename} ---")
    try:
        with open(filename, 'r') as f:
            for line in f:
                # Clean the word: remove whitespace and convert to lowercase
                word = line.strip().lower()
                if not word: # Skip any empty lines
                    continue

                stem, result = transduce(word)

                if result == "INVALID":
                    print(f"{word} -> Invalid Word")
                elif result is True: # is_plural
                    print(f"{word} -> {stem}+N+PL")
                else: # not plural (is singular)
                    print(f"{word} -> {word}+N+SG")

    except FileNotFoundError:
        print(f"\nError: The file '{filename}' was not found.")
        print(f"Please make sure '{filename}' is in the same directory as the script.")

# This makes the script runnable
if __name__ == "__main__":
    process_corpus()

--- FST Analysis of Nouns from brown_nouns.txt ---
investigation -> investigation+N+SG
primary -> primary+N+SG
election -> election+N+SG
evidence -> evidence+N+SG
irregularities -> irregularity+N+PL
place -> place+N+SG
jury -> jury+N+SG
presentments -> presentment+N+PL
charge -> charge+N+SG
election -> election+N+SG
praise -> praise+N+SG
thanks -> thank+N+PL
manner -> manner+N+SG
election -> election+N+SG
term -> term+N+SG
jury -> jury+N+SG
reports -> report+N+PL
irregularities -> irregularity+N+PL
primary -> primary+N+SG
handful -> handful+N+SG
reports -> report+N+PL
jury -> jury+N+SG
interest -> interest+N+SG
election -> election+N+SG
number -> number+N+SG
voters -> voter+N+PL
size -> size+N+SG
city -> city+N+SG
jury -> jury+N+SG
registration -> registration+N+SG
election -> election+N+SG
laws -> law+N+PL
legislators -> legislator+N+PL
laws -> law+N+PL
end -> end+N+SG
jury -> jury+N+SG
number -> number+N+SG
topics -> topic+N+PL
departments -> department+N+PL
practices -> practice+N+P